In [ ]:
import torch
from torch import nn as nn
from torch import optim as optim
import torch.utils.data as data
import math
import copy

In [ ]:
b = 10  # Example batch size
l = 20  # Example sequence length
d = 512 # Example embedding dimension

vector1 = torch.randn(b, l, d)
vector2 = torch.randn(b, l, d)

print("Vector 1 shape:", vector1.shape)
print("Vector 2 shape:", vector2.shape)

torch.matmul(vector1, vector2.transpose(-2, -1)).shape

Vector 1 shape: torch.Size([10, 20, 512])
Vector 2 shape: torch.Size([10, 20, 512])


torch.Size([10, 20, 20])

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)
        self.W_o = nn.Linear(d_model,d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1))/math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_scores = torch.softmax(attn_scores, dim = -1)
        output = torch.matmul(attn_scores, V)
        return output

    def split_heads(self, x):
        B, L, D = x.size()
        return x.view(B, L, self.num_heads, self.d_k).transpose(1,2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        attn_output = self.scaled_dot_product_attention(Q,K,V,mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [ ]:
torch.zeros(10, 12).unsqueeze(2).shape

torch.Size([10, 12, 1])

In [ ]:
torch.arange(0, 10, dtype=torch.float).shape

torch.Size([10])

In [ ]:
torch.arange(0,10,2).shape

torch.Size([5])

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, L):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(L, d_model)
        position = torch.arange(0, L, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(-1*torch.arange(0,d_model,2) * math.log(10000)/d_model).unsqueeze(0)
        pe[:,::2] = torch.sin(position/div_term)
        pe[:,1::2] = torch.cos(position/div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.multi_head_attention = MultiHeadAttention(d_model, num_heads)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.feedforward = PositionWiseFeedForward(d_model, d_ff)

    def forward(self, x, mask):
        attn_output = self.multi_head_attention(x,x,x,mask)
        x = self.norm1(x + self.dropout(attn_output))
        feed_output = self.feedforward(x)
        x = self.norm2(x + self.dropout(feed_output))
        return x

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt, device):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length, device = device), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt, device):
        src_mask, tgt_mask = self.generate_mask(src, tgt, device)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from collections import Counter

# --- 1. Load Multi30k EN→DE translation ---
dataset = load_dataset("bentrevett/multi30k")

# --- 2. Build vocabularies for both languages ---
PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"
SPECIAL_TOKENS = [PAD, UNK, SOS, EOS]

def build_vocab(dataset, key, min_freq=2):
    counter = Counter()
    for example in dataset:
        counter.update(example[key].lower().split())
    vocab = SPECIAL_TOKENS + [w for w, c in counter.items() if c >= min_freq]
    return {w: i for i, w in enumerate(vocab)}

en_vocab = build_vocab(dataset["train"], "en")
de_vocab = build_vocab(dataset["train"], "de")

# Reverse vocab for decoding predictions back to words
de_idx2word = {i: w for w, i in de_vocab.items()}

print(f"EN vocab: {len(en_vocab)} | DE vocab: {len(de_vocab)}")

# --- 3. Numericalize + add SOS/EOS tokens ---
def encode(sentence, vocab, max_len=50):
    tokens = sentence.lower().split()[:max_len - 2]
    ids = [vocab[SOS]]
    ids += [vocab.get(t, vocab[UNK]) for t in tokens]
    ids += [vocab[EOS]]
    return ids

def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for example in batch:
        src_batch.append(encode(example["en"], en_vocab))
        tgt_batch.append(encode(example["de"], de_vocab))

    # Pad to max length in batch
    src_max = max(len(s) for s in src_batch)
    tgt_max = max(len(t) for t in tgt_batch)

    src_padded = [s + [en_vocab[PAD]] * (src_max - len(s)) for s in src_batch]
    tgt_padded = [t + [de_vocab[PAD]] * (tgt_max - len(t)) for t in tgt_batch]

    return torch.tensor(src_padded), torch.tensor(tgt_padded)

train_loader = DataLoader(dataset["train"], batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(dataset["validation"], batch_size=64, collate_fn=collate_fn)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

EN vocab: 7704 | DE vocab: 9597


In [ ]:
# --- 4. Initialize Transformer ---
d_model = 256
num_heads = 8
num_layers = 3
d_ff = 512
max_seq_length = 60
dropout = 0.1

model = Transformer(
    src_vocab_size=len(en_vocab),
    tgt_vocab_size=len(de_vocab),
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_seq_length=max_seq_length,
    dropout=dropout
)

optimizer = optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=de_vocab[PAD])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Transformer(
  (encoder_embedding): Embedding(7704, 256)
  (decoder_embedding): Embedding(9597, 256)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-2): 3 x EncoderLayer(
      (multi_head_attention): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_features=256, bias=True)
        (W_v): Linear(in_features=256, out_features=256, bias=True)
        (W_o): Linear(in_features=256, out_features=256, bias=True)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (feedforward): PositionWiseFeedForward(
        (fc1): Linear(in_features=256, out_features=512, bias=True)
        (fc2): Linear(in_features=512, out_features=256, bias=True)
        (relu): ReLU()
      )
    )
  )
  (decoder_layers): ModuleList(
    (0-2): 3 x Decoder

In [ ]:
device

device(type='cuda')

In [ ]:
# --- 6. Validation loss ---
model.eval()
val_loss = 0

with torch.no_grad():
    for src, tgt in tqdm(val_loader, desc="Validation"):
        src, tgt = src.to(device), tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        output = model(src, tgt_input, device)
        loss = criterion(output.reshape(-1, len(de_vocab)), tgt_output.reshape(-1))
        val_loss += loss.item()

avg_val_loss = val_loss / len(val_loader)
print(f"Validation Loss: {avg_val_loss:.4f}")

Validation: 100%|██████████| 16/16 [00:00<00:00, 53.46it/s]

Validation Loss: 9.3225


In [ ]:
from tqdm import tqdm

# --- 5. Training loop ---
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)

    for src, tgt in loop:
        src, tgt = src.to(device), tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        output = model(src, tgt_input, device)

        loss = criterion(output.reshape(-1, len(de_vocab)), tgt_output.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

        # Update progress bar with running loss
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Avg Loss: {avg_loss:.4f}")

Epoch 1/30: 100%|██████████| 454/454 [00:16<00:00, 26.78it/s, loss=4.25]


Epoch 1/30 | Avg Loss: 5.3199


Epoch 2/30: 100%|██████████| 454/454 [00:16<00:00, 27.30it/s, loss=3.67]


Epoch 2/30 | Avg Loss: 4.0440


Epoch 3/30: 100%|██████████| 454/454 [00:17<00:00, 26.31it/s, loss=3.21]


...

Epoch 28/30 | Avg Loss: 1.2379


Epoch 29/30: 100%|██████████| 454/454 [00:17<00:00, 26.59it/s, loss=1.59]


Epoch 29/30 | Avg Loss: 1.1965


Epoch 30/30: 100%|██████████| 454/454 [00:17<00:00, 26.53it/s, loss=1.28]

Epoch 30/30 | Avg Loss: 1.1557


In [ ]:
# --- 6. Validation loss ---
model.eval()
val_loss = 0

with torch.no_grad():
    for src, tgt in tqdm(val_loader, desc="Validation"):
        src, tgt = src.to(device), tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        output = model(src, tgt_input, device)
        loss = criterion(output.reshape(-1, len(de_vocab)), tgt_output.reshape(-1))
        val_loss += loss.item()

avg_val_loss = val_loss / len(val_loader)
print(f"Validation Loss: {avg_val_loss:.4f}")

Validation: 100%|██████████| 16/16 [00:00<00:00, 86.61it/s]

Validation Loss: 2.0095


In [ ]:
# --- 6. Greedy decoding for inference ---
def translate(sentence, max_len=50):
    model.eval()
    src_ids = encode(sentence, en_vocab)
    src = torch.tensor([src_ids]).to(device)

    # Start decoder with <SOS>
    tgt_ids = [de_vocab[SOS]]

    for _ in range(max_len):
        tgt = torch.tensor([tgt_ids]).to(device)

        with torch.no_grad():
            output = model(src, tgt, device)

        next_token = output[0, -1, :].argmax().item()
        tgt_ids.append(next_token)

        if next_token == de_vocab[EOS]:
            break

    # Convert IDs back to words
    words = [de_idx2word.get(i, UNK) for i in tgt_ids[1:-1]]  # skip SOS/EOS
    return " ".join(words)

# --- 7. Test it ---
test_sentences = [
    "A man is walking a dog.",
    "Two children are playing in the park.",
    "A woman is sitting on a bench near the water."
]

for sent in test_sentences:
    print(f"EN: {sent}")
    print(f"DE: {translate(sent)}")
    print()

EN: A man is walking a dog.
DE: ein mann geht einen hund.

EN: Two children are playing in the park.
DE: zwei kinder spielen im park.

EN: A woman is sitting on a bench near the water.
DE: eine frau sitzt auf einer bank in der nähe des wassers.

